#  Veri Setinin İstatistiksel Olarak Dengeli Eğitim ve Test Kümelerine Ayrılması

Bu çalışma dosyasının amacı, makine öğrenmesi modelinin eğitileceği ve test edileceği veri kümelerini yapısal bütünlüğü korunacak şekilde oluşturmaktır. Kullanılan veri setinde "Yıl" değişkeni bulunmadığından dolayı kronolojik bir ayrım (Out-of-Time Validation) gerçekleştirilememiştir. Bu kısıtı telafi etmek ve özellik kaymasını (Covariate Shift) minimize etmek amacıyla, veri kümesi üzerinde hedef değişken tabakalandırması (Stratification) ve stokastik tohum optimizasyonu (Seed Optimization) metodolojileri uygulanmıştır.

In [1]:
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')

from scipy import stats
from scipy.stats import ks_2samp, anderson_ksamp, chi2_contingency

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics import roc_auc_score
from sklearn.ensemble import RandomForestClassifier

In [2]:
df = pd.read_csv("C:/Users/Asus/montesinho-fire-risk-prediction/data/processed/processed_forestfires_v2.csv")

print("Veri Boyutu:", df.shape)

df.head(10)
df.info()   
df.describe(include='all')

Veri Boyutu: (517, 23)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 517 entries, 0 to 516
Data columns (total 23 columns):
 #   Column                             Non-Null Count  Dtype  
---  ------                             --------------  -----  
 0   X                                  517 non-null    int64  
 1   Y                                  517 non-null    int64  
 2   FFMC                               517 non-null    float64
 3   DMC                                517 non-null    float64
 4   DC                                 517 non-null    float64
 5   ISI                                517 non-null    float64
 6   temp                               517 non-null    float64
 7   RH                                 517 non-null    int64  
 8   wind                               517 non-null    float64
 9   rain                               517 non-null    int64  
 10  area                               517 non-null    float64
 11  month_sin                          

,X,Y,FFMC,DMC,DC,ISI,temp,RH,wind,rain,...,day_sin,day_cos,is_weekend,is_peak_season,distance_to_center,dynamic_seasonal_hotspot_distance,moderate_wind_danger,hot_and_dry,double_drought,ISI_x_DC
count,517.000000,517.000000,517.000000,517.000000,517.000000,517.000000,517.000000,517.000000,517.000000,517.000000,...,5.170000e+02,517.000000,517.000000,517.000000,517.000000,517.000000,517.000000,517.000000,517.000000,517.000000
mean,4.669246,4.299807,90.644681,110.872340,547.940039,9.021663,18.889168,44.288201,4.017602,0.015474,...,-6.059765e-02,0.109757,0.346228,0.814313,2.456102,3.634859,0.473888,0.309478,0.682785,5202.014603
std,2.313778,1.229900,5.520111,64.046482,248.066192,4.559477,5.806625,16.317469,1.791653,0.123547,...,7.070413e-01,0.697339,0.476228,0.389230,1.192866,2.335392,0.499801,0.462726,0.465843,3253.018489
min,1.000000,2.000000,18.700000,1.100000,7.900000,0.000000,2.200000,15.000000,0.400000,0.000000,...,-9.749279e-01,-0.900969,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,3.000000,4.000000,90.200000,68.600000,437.700000,6.500000,15.500000,33.000000,2.700000,0.000000,...,-7.818315e-01,-0.222521,0.000000,1.000000,1.414214,2.236068,0.000000,0.000000,0.000000,3076.430000
50%,4.000000,4.000000,91.600000,108.300000,664.200000,8.400000,19.300000,42.000000,4.000000,0.000000,...,-2.449294e-16,-0.222521,0.000000,1.000000,2.236068,3.605551,0.000000,0.000000,1.000000,5471.250000
75%,7.000000,5.000000,92.900000,142.400000,713.900000,10.800000,22.800000,53.000000,4.900000,0.000000,...,7.818315e-01,0.623490,1.000000,1.000000,3.162278,5.385165,1.000000,1.000000,1.000000,6775.560000
max,9.000000,9.000000,96.200000,291.300000,860.600000,56.100000,33.300000,100.000000,9.400000,1.000000,...,9.749279e-01,1.000000,1.000000,1.000000,5.656854,8.062258,1.000000,1.000000,1.000000,16113.800000


## 1. Hedef Değişkenin Tabakalandırılması (Binning) ve Bileşik Sınıf Oluşturulması
Dengesiz dağılıma sahip (zero-inflated ve sağa çarpık) `area` değişkeninin eğitim ve test kümelerine orantılı dağıtılması, modelin genelleme yeteneği için kritiktir. 

Bu bağlamda şu adımlar izlenmiştir:
1. Yangın alanı gözlenmeyen (area = 0) kayıtlar ayrı bir sınıf olarak tanımlanmıştır.
2. Alanı 0'dan büyük olan gözlemler, dağılımı yansıtmak üzere 4 eşit çeyrekliğe (quantile) bölünmüştür.
3. Yangın şiddeti ile mevsimsel riskin eğitim/test ayrımında eşzamanlı olarak korunabilmesi amacıyla, oluşturulan alan sınıfları `is_peak_season` (zirve sezon) değişkeni ile birleştirilerek **Bileşik Hedef Sınıfı (Composite Target)** oluşturulmuştur.

In [3]:
df['target_bin'] = 0

positive_area_idx = df[df['area'] > 0].index

df.loc[positive_area_idx, 'target_bin'] = pd.qcut(df.loc[positive_area_idx, 'area'], q=4, labels=[1, 2, 3, 4])

df['composite_target'] = df['target_bin'].astype(str) + "_" + df['is_peak_season'].astype(str)

print("Sepetlere (Bin) Göre Yangın Sayıları:")
print(df['target_bin'].value_counts().sort_index())

print("\nBileşik Sınıf (YangınBoyutu_Sezon) Dağılımları:")
print(df['composite_target'].value_counts())

Sepetlere (Bin) Göre Yangın Sayıları:
target_bin
0    247
1     69
2     66
3     67
4     68
Name: count, dtype: int64

Bileşik Sınıf (YangınBoyutu_Sezon) Dağılımları:
composite_target
0_1    196
4_1     59
1_1     58
2_1     55
3_1     53
0_0     51
3_0     14
1_0     11
2_0     11
4_0      9
Name: count, dtype: int64


## 2. Seed Optimization ile Özellik Kaymasının Engellenmesi
Nispeten düşük örneklem büyüklüğüne (n=517) sahip veri setlerinde rastgele (random) yapılan veri ayrımları, bağımsız değişkenler üzerinde istatistiksel dengesizliklere yol açabilmektedir. Özellik kaymasını engellemek için özel bir stokastik arama algoritması tasarlanmıştır.

* Algoritma, 5000 farklı rastgele tohumu (random seed) tarayarak veri setini tabakalı olarak bölmektedir.
* Her bir senaryo için bağımsız değişkenlerin **PSI (Population Stability Index)** skorları hesaplanmaktadır. Herhangi bir değişkende PSI > 0.18 üzerinde kayma (drift) sergileyen tohumlar analiz dışı bırakılmıştır.
* Kriterleri sağlayan tohumlar arasından, değişkenler bazında **İki Örneklemli Kolmogorov-Smirnov (K-S)** testinin minimum p-değerini maksimize eden tohum, optimum ayırıcı olarak belirlenmektedir.

In [9]:
continuous_features = ['temp', 'wind', 'RH', 'FFMC', 'DMC', 'DC', 'ISI', 'distance_to_center', 'X', 'Y']

X_data = df.drop(columns=['target_bin', 'composite_target'])
y_composite = df['composite_target']

def get_max_psi(expected, actual, buckets=10):
    max_psi = 0
    for col in expected.columns:
        e_array, a_array = expected[col].values, actual[col].values
        
        breakpoints = np.arange(0, buckets + 1) / (buckets) * 100
        min_val, max_val = np.min(e_array), np.max(e_array)
        
        bp = breakpoints.copy()
        bp += -(np.min(bp))
        bp /= (np.max(bp) / (max_val - min_val))
        bp += min_val
        
        e_frac = np.histogram(e_array, bp)[0] / len(e_array)
        a_frac = np.histogram(a_array, bp)[0] / len(a_array)
        
        e_frac = np.where(e_frac == 0, 0.0001, e_frac)
        a_frac = np.where(a_frac == 0, 0.0001, a_frac)
        
        col_psi = np.sum((e_frac - a_frac) * np.log(e_frac / a_frac))
        if col_psi > max_psi:
            max_psi = col_psi
    return max_psi

best_seed = 0
best_min_p = -1.0
best_mean_p = -1.0
best_max_psi = 100.0
iterations = 5000 

for seed in range(1, iterations + 1):
    X_train_temp, X_test_temp = train_test_split(X_data, test_size=0.20, random_state=seed, stratify=y_composite)
    
    max_psi_val = get_max_psi(X_train_temp[continuous_features], X_test_temp[continuous_features])
    if max_psi_val > 0.18:
        continue 
        
    p_values = [ks_2samp(X_train_temp[feat], X_test_temp[feat])[1] for feat in continuous_features]
    min_p = np.min(p_values)
    mean_p = np.mean(p_values)
    
    if min_p > best_min_p:
        best_min_p = min_p
        best_mean_p = mean_p
        best_max_psi = max_psi_val
        best_seed = seed
    elif min_p == best_min_p and mean_p > best_mean_p:
        best_mean_p = mean_p
        best_seed = seed

if best_seed != 0:
    print(f"Mükemmel Dengeyi Kuran 'Ultimate Golden Seed': {best_seed}")
    print(f"Bu tohumdaki EN KÖTÜ PSI Skoru: {best_max_psi:.4f} (Güvenli Bölgede!)")
    print(f"En Düşük K-S p-değeri (Zayıf Halka): {best_min_p:.4f}")
    print(f"Ortalama p-değeri: {best_mean_p:.4f}")
else:
    print("İmkansız şartlar! Bu kadar dar veride hem PSI < 0.18 olsun hem de eşit dağılsın diyen bir tohum 5000 denemede çıkmadı.")

Mükemmel Dengeyi Kuran 'Ultimate Golden Seed': 4601
Bu tohumdaki EN KÖTÜ PSI Skoru: 0.1124 (Güvenli Bölgede!)
En Düşük K-S p-değeri (Zayıf Halka): 0.6816
Ortalama p-değeri: 0.9068


## 3. Data Splitting
Optimizasyon algoritması tarafından seçilen optimum tohum kullanılarak, veri seti %80 Eğitim (Train) ve %20 Test olacak şekilde tabakalı yaklaşımla ikiye ayrılmıştır.

In [15]:
best_seed = 4601
X_train_strat, X_test_strat = train_test_split(X_data, test_size=0.20, random_state=best_seed, stratify=y_composite)

print(f"Veri Seti 'Golden Seed' ({best_seed}) kullanılarak Train ve Test olarak kusursuzca ayrıldı!")
print(f"Train Boyutu: {X_train_strat.shape}")
print(f"Test Boyutu: {X_test_strat.shape}")

Veri Seti 'Golden Seed' (4601) kullanılarak Train ve Test olarak kusursuzca ayrıldı!
Train Boyutu: (413, 23)
Test Boyutu: (104, 23)


## 4. İstatistiksel Kalite Kontrolü: PSI ve Mekansal Dağılım Analizi
Gerçekleştirilen veri ayrımının yapısal bütünlüğü iki farklı istatistiksel yaklaşımla test edilmiştir:
1. **PSI Analizi:** Her bir değişken için Eğitim ve Test kümeleri arasındaki dağılım sapması hesaplanmıştır. PSI değerinin 0.1'in altında olması, iki küme arasında istatistiksel olarak anlamlı bir özellik kayması olmadığını ifade etmektedir.
2. **Mekansal Dağılım Testi:** Yangınların koordinat düzlemindeki (X ve Y eksenleri) konumları K-S testleri ile incelenmiş, modelin mekansal olarak yansız (unbiased) bir dağılım ile eğitilmesi hedeflenmiştir.

In [16]:
def calculate_psi(expected, actual, buckets=10):
    
    def psi(expected_array, actual_array, buckets):
        def scale_range (input, min, max):
            input += -(np.min(input))
            input /= np.max(input) / (max - min)
            input += min
            return input
        breakpoints = np.arange(0, buckets + 1) / (buckets) * 100
        breakpoints = scale_range(breakpoints, np.min(expected_array), np.max(expected_array))
        expected_fractions = np.histogram(expected_array, breakpoints)[0] / len(expected_array)
        actual_fractions = np.histogram(actual_array, breakpoints)[0] / len(actual_array)
        
        def sub_psi(e_perc, a_perc):
            if a_perc == 0: a_perc = 0.0001
            if e_perc == 0: e_perc = 0.0001
            return (e_perc - a_perc) * np.log(e_perc / a_perc)
        
        return np.sum(sub_psi(expected_fractions[i], actual_fractions[i]) for i in range(len(expected_fractions)))
    
    psi_values = []
    for feature in expected.columns:
        psi_values.append(psi(expected[feature].values, actual[feature].values, buckets))
    return pd.Series(psi_values, index=expected.columns)

print("===  PSI SKORLARI ===")
print("(Not: Bankacılık standardında PSI < 0.1 ise kayma yoktur ve mükemmeldir)\n")
psi_scores = calculate_psi(X_train_strat[continuous_features], X_test_strat[continuous_features])
for feat, score in psi_scores.items():
    status = "MÜKEMMEL" if score < 0.1 else ("KABUL EDİLEBİLİR" if score < 0.2 else "KAYMA VAR!")
    print(f"{feat:20}: {score:.4f} -> {status}")

print("\n===  K-S TESTLERİ ===")
_, p_x = ks_2samp(X_train_strat['X'], X_test_strat['X'])
_, p_y = ks_2samp(X_train_strat['Y'], X_test_strat['Y'])
print(f"X Koordinatı K-S p-değeri: {p_x:.4f} (Beklenti > 0.05)")
print(f"Y Koordinatı K-S p-değeri: {p_y:.4f} (Beklenti > 0.05)")

===  PSI SKORLARI ===
(Not: Bankacılık standardında PSI < 0.1 ise kayma yoktur ve mükemmeldir)

temp                : 0.0398 -> MÜKEMMEL
wind                : 0.0819 -> MÜKEMMEL
RH                  : 0.1089 -> KABUL EDİLEBİLİR
FFMC                : 0.0548 -> MÜKEMMEL
DMC                 : 0.0920 -> MÜKEMMEL
DC                  : 0.0478 -> MÜKEMMEL
ISI                 : 0.0498 -> MÜKEMMEL
distance_to_center  : 0.1124 -> KABUL EDİLEBİLİR
X                   : 0.0682 -> MÜKEMMEL
Y                   : 0.0298 -> MÜKEMMEL

===  K-S TESTLERİ ===
X Koordinatı K-S p-değeri: 0.9719 (Beklenti > 0.05)
Y Koordinatı K-S p-değeri: 0.9999 (Beklenti > 0.05)


## 5.  Adversarial Validation ve Permütasyon Testi
Eğitim ve test kümelerinin istatistiksel dağılımları itibarıyla birbirinden ayırt edilemez olduğu varsayımını test etmek için "Adversarial Validation" (Düşman Doğrulama) çerçevesi kullanılmıştır.
* Eğitim verilerine 0, Test verilerine 1 bağımlı değişken etiketleri atanarak bir sınıflandırıcı (RandomForestClassifier) eğitilmiştir.
* Modelin iki kümeyi ayırt etme gücü, Çapraz Doğrulama (Cross-Validation) yaklaşımıyla ROC-AUC metriği üzerinden ölçülmüştür. AUC skorunun 0.50 eşik değerine yakınsaması, dağılımlar arası belirsizliği ifade eder.
* Modelin elde ettiği AUC skorunun şans eseri olup olmadığını değerlendirmek üzere 100 iterasyonluk **Permütasyon Testi** uygulanmıştır. Elde edilen p-değerinin (p > 0.05) anlamlılık sınırının üzerinde olması, eğitim ve test setleri arasında istatistiksel olarak anlamlı bir farklılık bulunmadığı hipotezini desteklemektedir.

In [17]:
X_train_adv = X_train_strat.copy()
X_test_adv = X_test_strat.copy()

X_train_adv['target_is_test'] = 0
X_test_adv['target_is_test'] = 1

df_adv = pd.concat([X_train_adv, X_test_adv], axis=0).reset_index(drop=True)
X_adv = df_adv.drop(columns=['target_is_test'])
y_adv = df_adv['target_is_test']

clf = RandomForestClassifier(n_estimators=100, random_state=42, max_depth=3)

cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
scores = cross_val_score(clf, X_adv, y_adv, cv=cv, scoring='roc_auc')
auc_score = scores.mean()

print(f"Adversarial Validation AUC Skoru: {auc_score:.4f} (Beklenti: ~0.50)")

n_permutations = 100
better_scores = 0

for i in range(n_permutations):
    y_perm = np.random.permutation(y_adv)
    perm_scores = cross_val_score(clf, X_adv, y_perm, cv=cv, scoring='roc_auc')
    if perm_scores.mean() >= auc_score:
        better_scores += 1

p_value = (better_scores + 1) / (n_permutations + 1)
print(f"Permütasyon Testi p-değeri: {p_value:.4f}")

Adversarial Validation AUC Skoru: 0.4151 (Beklenti: ~0.50)
Permütasyon Testi p-değeri: 0.9901


In [18]:
save_dir = '../../data/processed'
if not os.path.exists(save_dir):
    os.makedirs(save_dir)

X_train_strat.to_csv(f'{save_dir}/X_train_v2.csv', index=False)
X_test_strat.to_csv(f'{save_dir}/X_test_v2.csv', index=False)